# ToS-DETR External Inference Demo

This notebook runs ToS-DETR on de-identified 2.5D example images and aligned cartilage masks. The sample files verify the inference and visualization pipeline.


## 1. Install dependencies

Run this cell in a fresh Python 3.11 environment. `requirements.txt` pins the Python package versions used for the release. The MSDeformAttn extension also requires system-level CUDA build tools: an NVIDIA GPU, a CUDA-enabled PyTorch build, CUDA Toolkit with `nvcc`, and a compatible C++ compiler such as `g++`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
ops_so = list((REPO_ROOT / 'models' / 'ops').glob('**/*.so'))
if not ops_so:
    env = os.environ.copy()
    env.setdefault('PYTHON_BIN', sys.executable)
    subprocess.run(['bash', 'make.sh'], cwd=REPO_ROOT / 'models' / 'ops', env=env, check=True)
else:
    print('MSDeformAttn extension already built:', ops_so[0])


## 2. Configure paths and checkpoint

Download `checkpoint_best_medical.pth` from the ToS-DETR Hugging Face repository, then set `CHECKPOINT_PATH`. Do not commit checkpoints to git.

- Model page: <https://huggingface.co/SL3w/ToS-DETR>
- Direct download: <https://huggingface.co/SL3w/ToS-DETR/resolve/main/checkpoint_best_medical.pth>

Download command:

```bash
scripts/download_checkpoint.sh --output weights/checkpoint_best_medical.pth
```


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

CHECKPOINT_PATH = REPO_ROOT / 'weights' / 'checkpoint_best_medical.pth'
IMAGE_DIR = REPO_ROOT / 'sample_data' / 'images'
MASK_DIR = REPO_ROOT / 'sample_data' / 'masks'
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'demo_inference'

print('repo:', REPO_ROOT)
print('sample images:', sorted(p.name for p in IMAGE_DIR.glob('*.png')))
print('sample masks:', sorted(p.name for p in MASK_DIR.glob('*.png')))

if not CHECKPOINT_PATH.exists():
    env = os.environ.copy()
    env.setdefault('PYTHON_BIN', sys.executable)
    subprocess.run([
        'bash', str(REPO_ROOT / 'scripts' / 'download_checkpoint.sh'),
        '--output', str(CHECKPOINT_PATH),
    ], cwd=REPO_ROOT, env=env, check=True)

print('checkpoint exists:', CHECKPOINT_PATH.exists(), CHECKPOINT_PATH)


## 3. Run inference

The command writes `summary.json`, case predictions, positive-case labels, visualized positive images, and threshold-filtered box JSON. If the checkpoint is missing, the cell prints the command that should be run after placing the weight file.


In [ ]:
import os
import subprocess
import sys

cmd = [
    'bash', str(REPO_ROOT / 'scripts' / 'run_external_infer.sh'),
    '--checkpoint', str(CHECKPOINT_PATH),
    '--image-dir', str(IMAGE_DIR),
    '--mask-dir', str(MASK_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--max-images', '2',
]
env = os.environ.copy()
env.setdefault('DEVICE', 'cuda:0')
env.setdefault('BATCH_SIZE', '2')
env.setdefault('NUM_WORKERS', '1')
env.setdefault('PYTHON_BIN', sys.executable)

if CHECKPOINT_PATH.exists():
    subprocess.run(cmd, cwd=REPO_ROOT, env=env, check=True)
else:
    print('Checkpoint not found. Place checkpoint_best_medical.pth at:', CHECKPOINT_PATH)
    print('Command to run after downloading the checkpoint:')
    print(' '.join(cmd))


## 4. Inspect outputs


In [ ]:
import json
from IPython.display import display
from PIL import Image

summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
    vis_dir = OUTPUT_DIR / 'positive_case_images'
    vis_images = sorted(vis_dir.glob('*.png'))
    if vis_images:
        for path in vis_images[:4]:
            print(path.name)
            display(Image.open(path))
    else:
        print('No positive visualizations were produced for this checkpoint and sample data.')
else:
    print('No summary.json yet. Run inference after setting CHECKPOINT_PATH.')
